# Chapter 05 — SOLUTIONS: Classification Models

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid')
np.random.seed(42)

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Setup complete!')

## Task 1 Solution: Logistic Regression Baseline

In [ ]:
lr = Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))])
lr.fit(X_train, y_train)
print(f'Logistic Regression Accuracy: {accuracy_score(y_test, lr.predict(X_test)):.4f}')

## Task 2 Solution: KNN Classifier

In [ ]:
knn = Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=7))])
knn.fit(X_train, y_train)
print(f'KNN (k=7) Accuracy: {accuracy_score(y_test, knn.predict(X_test)):.4f}')

## Task 3 Solution: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f'Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=wine.target_names))

## Task 4 Solution: Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm, display_labels=wine.target_names).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Random Forest — Confusion Matrix (Wine Dataset)', fontsize=12)
plt.tight_layout()
plt.show()
print('Most confusion happens between class_1 and class_2.')
print('Class_0 (high quality) is nearly perfectly separated — very distinct wines!')

## Task 5 Solution: Cross-Validation Comparison

In [ ]:
models = {
    'Logistic Regression': lr,
    'KNN (k=7)':           knn,
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       rf,
}

print('5-Fold Cross-Validation (Accuracy):')
print(f'{"Model":<22}  {"Mean":>8}  {"Std":>6}')
print('-' * 40)

cv_means, cv_stds = {}, {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_means[name] = scores.mean()
    cv_stds[name]  = scores.std()
    print(f'{name:<22}  {scores.mean():>8.4f}  ±{scores.std():.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
names = list(cv_means.keys())
means = [cv_means[n] for n in names]
stds  = [cv_stds[n]  for n in names]
ax.barh(names, means, xerr=stds, color='#3498db', alpha=0.75, capsize=5)
ax.set_xlabel('Accuracy (5-Fold CV)')
ax.set_title('Model Comparison: Cross-Validated Accuracy')
ax.set_xlim(0.85, 1.02)
plt.tight_layout()
plt.show()

best = max(cv_means, key=cv_means.get)
print(f'\nBest model: {best}')

## Bonus Solution: Support Vector Machine

In [ ]:
svm = Pipeline([('s', StandardScaler()), ('m', SVC(kernel='rbf', C=1.0))])
svm_scores = cross_val_score(svm, X, y, cv=5, scoring='accuracy')

print(f'SVM (RBF kernel):      {svm_scores.mean():.4f} ± {svm_scores.std():.4f}')
print()
print('Comparison with other models (from Task 5):')
for name, model in [('Logistic Regression', lr), ('KNN (k=7)', knn), ('Random Forest', rf)]:
    s = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'{name:<22}  {s.mean():.4f} ± {s.std():.4f}')

print('\n→ SVM with RBF kernel often achieves top accuracy on this dataset.')
print('  It finds a non-linear boundary in the high-dimensional feature space.')